In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('2025_LoL_esports_match_data_from_OraclesElixir.csv', low_memory=False)
print(df.shape)

(120456, 165)


In [2]:
major_leagues = ['LCK', 'LPL', 'LEC', 'LCS', 'LTA N', 'LTA S']

team_df = df[
    (df['position'] == 'team') &
    (df['league'].isin(major_leagues))
].copy()

print(team_df.shape)
print(team_df['league'].value_counts())

(4204, 165)
league
LPL      1610
LCK      1110
LEC       612
LTA S     444
LTA N     428
Name: count, dtype: int64


In [3]:
cols = [
    'gameid', 'league', 'year', 'split', 'playoffs',
    'teamname', 'side', 'result',
    'ban1', 'ban2', 'ban3', 'ban4', 'ban5',
    'pick1', 'pick2', 'pick3', 'pick4', 'pick5'
]

team_df = team_df[cols]
team_df.head()

,gameid,league,year,split,playoffs,teamname,side,result,ban1,ban2,ban3,ban4,ban5,pick1,pick2,pick3,pick4,pick5
58,11715-11715_game_1,LPL,2025,Split 1,0,Weibo Gaming,Blue,1,Jayce,Poppy,Rumble,Rakan,Rell,Ashe,Aurora,Braum,K'Sante,Viego
59,11715-11715_game_1,LPL,2025,Split 1,0,Oh My God,Red,0,Skarner,Vi,Kalista,Xin Zhao,Wukong,Varus,Nocturne,Orianna,Neeko,Gnar
70,11715-11715_game_2,LPL,2025,Split 1,0,Oh My God,Blue,0,Caitlyn,Vi,Xin Zhao,Poppy,Alistar,Rumble,Rell,Jhin,Maokai,Ambessa
71,11715-11715_game_2,LPL,2025,Split 1,0,Weibo Gaming,Red,1,Kalista,Skarner,Corki,Sejuani,Yone,Jayce,Miss Fortune,Wukong,Rakan,Viktor
94,11715-11715_game_3,LPL,2025,Split 1,0,Oh My God,Blue,0,Vi,Caitlyn,Xin Zhao,Akali,Alistar,Leona,Zyra,Yone,Ziggs,Aatrox


In [4]:
team_df = team_df.dropna(subset=['pick1','pick2','pick3','pick4','pick5'])

print('Games (team rows):', len(team_df))
print('Unique games:', team_df['gameid'].nunique())
print('Expected ratio (should be ~2):', len(team_df) / team_df['gameid'].nunique())

Games (team rows): 4204
Unique games: 2102
Expected ratio (should be ~2): 2.0


In [5]:
blue = team_df[team_df['side'] == 'Blue'].add_prefix('blue_').rename(columns={'blue_gameid': 'gameid'})
red = team_df[team_df['side'] == 'Red'].add_prefix('red_').rename(columns={'red_gameid': 'gameid'})

games = blue.merge(red, on='gameid')

# result: 1 = blue wins, 0 = red wins
games['blue_win'] = games['blue_result'].astype(int)

print('Final games shape:', games.shape)
games.head()

Final games shape: (2102, 36)


,gameid,blue_league,blue_year,blue_split,blue_playoffs,blue_teamname,blue_side,blue_result,blue_ban1,blue_ban2,...,red_ban2,red_ban3,red_ban4,red_ban5,red_pick1,red_pick2,red_pick3,red_pick4,red_pick5,blue_win
0,11715-11715_game_1,LPL,2025,Split 1,0,Weibo Gaming,Blue,1,Jayce,Poppy,...,Vi,Kalista,Xin Zhao,Wukong,Varus,Nocturne,Orianna,Neeko,Gnar,1
1,11715-11715_game_2,LPL,2025,Split 1,0,Oh My God,Blue,0,Caitlyn,Vi,...,Skarner,Corki,Sejuani,Yone,Jayce,Miss Fortune,Wukong,Rakan,Viktor,0
2,11715-11715_game_3,LPL,2025,Split 1,0,Oh My God,Blue,0,Vi,Caitlyn,...,Kalista,Corki,Kai'Sa,Renekton,Jax,Sejuani,Jinx,Poppy,Sylas,0
3,11716-11716_game_1,LPL,2025,Split 1,0,LNG Esports,Blue,1,Galio,Nocturne,...,Ashe,Renata Glasc,Akali,Viktor,Rumble,Vi,Aurora,Ezreal,Leona,1
4,11716-11716_game_2,LPL,2025,Split 1,0,ThunderTalk Gaming,Blue,1,Jayce,Renata Glasc,...,Ashe,Varus,Miss Fortune,Gragas,Corki,Rell,Taliyah,Renekton,Lee Sin,1


In [6]:
print('Blue win rate:', games['blue_win'].mean().round(3))
print('Null values:', games.isnull().sum().sum())
print('Leagues:', games['blue_league'].value_counts())

Blue win rate: 0.527
Null values: 126
Leagues: blue_league
LPL      805
LCK      555
LEC      306
LTA S    222
LTA N    214
Name: count, dtype: int64


In [7]:
null_cols = games.isnull().sum()
print(null_cols[null_cols > 0])

blue_split    53
blue_ban3      3
blue_ban4      3
blue_ban5      4
red_split     53
red_ban3       1
red_ban4       6
red_ban5       3
dtype: int64


In [8]:
games['blue_split'] = games['blue_split'].fillna('Unknown')
games['red_split'] = games['red_split'].fillna('Unknown')

ban_cols = ['blue_ban3','blue_ban4','blue_ban5','red_ban3','red_ban4','red_ban5']
games = games.dropna(subset=ban_cols)

print('Shape after cleaning:', games.shape)
print('Null values remaining:', games.isnull().sum().sum())

Shape after cleaning: (2087, 36)
Null values remaining: 0


In [9]:
pick_cols_blue = ['blue_pick1','blue_pick2','blue_pick3','blue_pick4','blue_pick5']
pick_cols_red = ['red_pick1','red_pick2','red_pick3','red_pick4','red_pick5']

all_picks = pd.Series(
    games[pick_cols_blue].values.flatten().tolist() + 
    games[pick_cols_red].values.flatten().tolist()
)
pick_counts = all_picks.value_counts()

print('Top 20 most picked champions:')
print(pick_counts.head(20))

Top 20 most picked champions:
Xin Zhao    618
Rell        574
Alistar     567
Rumble      550
Taliyah     517
Ambessa     495
Rakan       474
Corki       474
Varus       472
Ezreal      459
Nautilus    429
Kai'Sa      420
Vi          411
Azir        409
Wukong      401
Aurora      388
Sion        379
Sejuani     377
Leona       364
Braum       358
Name: count, dtype: int64


In [10]:
records = []
for _, row in games.iterrows():
    for col in pick_cols_blue:
        records.append({'champion': row[col], 'win': row['blue_win']})
    for col in pick_cols_red:
        records.append({'champion': row[col], 'win': 1 - row['blue_win']})

champ_df = pd.DataFrame(records)
champ_stats = champ_df.groupby('champion').agg(
    games=('win','count'),
    wins=('win','sum')
).reset_index()
champ_stats['winrate'] = champ_stats['wins'] / champ_stats['games']

# filter to champs with at least 20 games
champ_stats = champ_stats[champ_stats['games'] >= 20].sort_values('winrate', ascending=False)
print('Top 15 win rate champions (min 20 games):')
print(champ_stats.head(15).to_string())
print('\nBottom 15:')
print(champ_stats.tail(15).to_string())

Top 15 win rate champions (min 20 games):
       champion  games  wins   winrate
56       Kennen     23    15  0.652174
20      Camille     74    45  0.608108
43        Ivern     32    19  0.593750
89        Poppy    315   186  0.590476
52      Kalista    177   104  0.587571
19      Caitlyn     75    44  0.586667
82      Nidalee    113    66  0.584071
116      Syndra     23    13  0.565217
144        Yone    286   161  0.562937
97     Renekton    276   155  0.561594
45    Jarvan IV    180   101  0.561111
16   Blitzcrank     43    24  0.558140
8         Annie    225   125  0.555556
81        Neeko    316   175  0.553797
92        Rakan    474   262  0.552743

Bottom 15:
      champion  games  wins   winrate
135     Viktor    300   134  0.446667
154       Zyra     79    35  0.443038
78        Nami     61    27  0.442623
14        Bard    138    61  0.442029
80    Nautilus    429   189  0.440559
63       Leona    364   160  0.439560
134      Viego    124    54  0.435484
27      Draven    

In [11]:
league_stats = games.groupby('blue_league')['blue_win'].agg(['mean','count']).reset_index()
league_stats.columns = ['league','blue_winrate','games']
print(league_stats.sort_values('blue_winrate', ascending=False).to_string())

  league  blue_winrate  games
1    LEC      0.559603    302
2    LPL      0.529338    801
4  LTA S      0.516129    217
3  LTA N      0.511737    213
0    LCK      0.510830    554


In [12]:
from itertools import combinations

pair_counts = {}
pair_wins = {}

for _, row in games.iterrows():
    for side, win in [('blue', row['blue_win']), ('red', 1 - row['blue_win'])]:
        picks = [row[f'{side}_pick{i}'] for i in range(1, 6)]
        for pair in combinations(sorted(picks), 2):
            pair_counts[pair] = pair_counts.get(pair, 0) + 1
            pair_wins[pair] = pair_wins.get(pair, 0) + win

pair_df = pd.DataFrame({
    'pair': list(pair_counts.keys()),
    'games': list(pair_counts.values()),
    'wins': list(pair_wins.values())
})
pair_df['winrate'] = pair_df['wins'] / pair_df['games']
pair_df = pair_df[pair_df['games'] >= 15].sort_values('games', ascending=False)

print('Most frequent pairs:')
print(pair_df.head(15).to_string())
print('\nHighest winrate pairs (min 15 games):')
print(pair_df.sort_values('winrate', ascending=False).head(15).to_string())

Most frequent pairs:
                         pair  games  wins   winrate
227            (Rakan, Xayah)    164   101  0.615854
299       (Taliyah, Xin Zhao)    124    69  0.556452
310           (Sejuani, Yone)    112    62  0.553571
90              (Corki, Rell)    110    49  0.445455
1414        (Alistar, Rumble)    108    66  0.611111
29             (Rell, Rumble)     98    56  0.571429
333         (Rakan, Xin Zhao)     96    51  0.531250
658        (Alistar, Taliyah)     96    51  0.531250
2947          (Braum, Lucian)     94    47  0.500000
97            (Rell, Taliyah)     89    45  0.505618
74            (Ezreal, Leona)     89    38  0.426966
116         (Varus, Xin Zhao)     87    43  0.494253
796        (Rumble, Xin Zhao)     86    47  0.546512
262   (Kalista, Renata Glasc)     86    49  0.569767
650          (Alistar, Varus)     86    56  0.651163

Highest winrate pairs (min 15 games):
                         pair  games  wins   winrate
3158           (Annie, Rakan)     25   

In [13]:
games.to_csv('lol_games_cleaned.csv', index=False)
print('Saved!')

Saved!


In [14]:
from sklearn.preprocessing import MultiLabelBinarizer

pick_cols_blue = ['blue_pick1','blue_pick2','blue_pick3','blue_pick4','blue_pick5']
pick_cols_red = ['red_pick1','red_pick2','red_pick3','red_pick4','red_pick5']

# combine all 10 picks per game into a list
games['all_picks'] = games[pick_cols_blue + pick_cols_red].values.tolist()

mlb = MultiLabelBinarizer()
X_picks = mlb.fit_transform(games['all_picks'])

# add blue side as a feature (1 = we're predicting blue win, blue has side advantage)
X = np.hstack([X_picks, games['blue_win'].values.reshape(-1,1)*0 + 1])  
# actually just use picks for now, side is the label not a feature
X = X_picks

y = games['blue_win'].values

print('Feature matrix shape:', X.shape)
print('Label distribution:', y.mean().round(3))
print('Champions encoded:', len(mlb.classes_))

Feature matrix shape: (2087, 155)
Label distribution: 0.526
Champions encoded: 155


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

Train size: 1669
Test size: 418


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': BernoulliNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results[name] = {'accuracy': round(acc, 4), 'f1': round(f1, 4)}
    print(f'{name}: Accuracy={acc:.4f}, F1={f1:.4f}')

Logistic Regression: Accuracy=0.4737, F1=0.5238
Naive Bayes: Accuracy=0.4641, F1=0.5410
Random Forest: Accuracy=0.5335, F1=0.5895


In [17]:
# add blue side advantage as explicit feature (always 1 for our framing)
# more importantly add it as a neutral feature — let's encode side properly
# since our matrix is always from blue's perspective, add a constant
# actually let's try adding patch/league as features too

from sklearn.preprocessing import LabelEncoder

side_feature = np.ones((len(X), 1))  # always predicting from blue perspective

league_enc = LabelEncoder()
league_feature = league_enc.fit_transform(games['blue_league']).reshape(-1, 1)

X_with_meta = np.hstack([X_picks, side_feature, league_feature])

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_with_meta, y, test_size=0.2, random_state=42
)

for name, model in models.items():
    model.fit(X_train2, y_train2)
    preds = model.predict(X_test2)
    acc = accuracy_score(y_test2, preds)
    f1 = f1_score(y_test2, preds)
    print(f'{name} + meta: Accuracy={acc:.4f}, F1={f1:.4f}')

Logistic Regression + meta: Accuracy=0.4713, F1=0.5227
Naive Bayes + meta: Accuracy=0.4665, F1=0.5421
Random Forest + meta: Accuracy=0.4833, F1=0.5443


In [18]:
import subprocess
subprocess.run(['pip', 'install', 'node2vec', '--quiet'])
print('done')

done


In [19]:
import networkx as nx
from itertools import combinations

G = nx.Graph()

for _, row in games.iterrows():
    blue_picks = [row[f'blue_pick{i}'] for i in range(1,6)]
    red_picks = [row[f'red_pick{i}'] for i in range(1,6)]
    
    # same team = synergy edges
    for pair in combinations(blue_picks, 2):
        if G.has_edge(*pair):
            G[pair[0]][pair[1]]['weight'] += 1
        else:
            G.add_edge(pair[0], pair[1], weight=1)
    
    for pair in combinations(red_picks, 2):
        if G.has_edge(*pair):
            G[pair[0]][pair[1]]['weight'] += 1
        else:
            G.add_edge(pair[0], pair[1], weight=1)

print('Nodes (champions):', G.number_of_nodes())
print('Edges (co-occurrence pairs):', G.number_of_edges())

Nodes (champions): 155
Edges (co-occurrence pairs): 4468


In [20]:
from node2vec import Node2Vec

node2vec = Node2Vec(
    G, 
    dimensions=64, 
    walk_length=20, 
    num_walks=200, 
    workers=1,
    quiet=True
)

model_n2v = node2vec.fit(window=10, min_count=1)
print('Embedding dimensions:', model_n2v.wv.vector_size)
print('Sample embedding for Xin Zhao:', model_n2v.wv['Xin Zhao'][:5])

Embedding dimensions: 64
Sample embedding for Xin Zhao: [ 0.14112249 -0.08066146  0.08593375  0.26723537  0.04253649]


In [21]:
def team_embedding(picks, model):
    return np.sum([model.wv[p] for p in picks], axis=0)

blue_embeddings = np.array([
    team_embedding([row[f'blue_pick{i}'] for i in range(1,6)], model_n2v)
    for _, row in games.iterrows()
])

red_embeddings = np.array([
    team_embedding([row[f'red_pick{i}'] for i in range(1,6)], model_n2v)
    for _, row in games.iterrows()
])

# concatenate blue and red team embeddings
X_emb = np.hstack([blue_embeddings, red_embeddings])
print('Embedding feature matrix shape:', X_emb.shape)

Embedding feature matrix shape: (2087, 128)


In [22]:
from sklearn.ensemble import GradientBoostingClassifier

X_train_emb, X_test_emb, y_train_emb, y_test_emb = train_test_split(
    X_emb, y, test_size=0.2, random_state=42
)

emb_models = {
    'Logistic Regression + Embeddings': LogisticRegression(max_iter=1000),
    'Random Forest + Embeddings': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting + Embeddings': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for name, model in emb_models.items():
    model.fit(X_train_emb, y_train_emb)
    preds = model.predict(X_test_emb)
    acc = accuracy_score(y_test_emb, preds)
    f1 = f1_score(y_test_emb, preds)
    print(f'{name}: Accuracy={acc:.4f}, F1={f1:.4f}')

Logistic Regression + Embeddings: Accuracy=0.5120, F1=0.5732
Random Forest + Embeddings: Accuracy=0.5120, F1=0.5732
Gradient Boosting + Embeddings: Accuracy=0.5000, F1=0.5708


In [23]:
print('='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
print(f'Baseline - Logistic Regression:     Acc=0.4737, F1=0.5238')
print(f'Baseline - Naive Bayes:             Acc=0.4641, F1=0.5410')
print(f'Baseline - Random Forest:           Acc=0.5335, F1=0.5895')
print('='*60)

FINAL RESULTS SUMMARY
Baseline - Logistic Regression:     Acc=0.4737, F1=0.5238
Baseline - Naive Bayes:             Acc=0.4641, F1=0.5410
Baseline - Random Forest:           Acc=0.5335, F1=0.5895


In [24]:
# difference captures counter relationships between teams
X_emb_diff = np.hstack([blue_embeddings, red_embeddings, blue_embeddings - red_embeddings])
print('Shape with diff:', X_emb_diff.shape)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_emb_diff, y, test_size=0.2, random_state=42
)

for name, model in emb_models.items():
    model.fit(X_train_d, y_train_d)
    preds = model.predict(X_test_d)
    acc = accuracy_score(y_test_d, preds)
    f1 = f1_score(y_test_d, preds)
    print(f'{name} + diff: Accuracy={acc:.4f}, F1={f1:.4f}')

Shape with diff: (2087, 192)
Logistic Regression + Embeddings + diff: Accuracy=0.5167, F1=0.5756
Random Forest + Embeddings + diff: Accuracy=0.5215, F1=0.5781
Gradient Boosting + Embeddings + diff: Accuracy=0.5000, F1=0.5708


In [25]:
from node2vec import Node2Vec

node2vec_128 = Node2Vec(G, dimensions=128, walk_length=30, num_walks=300, workers=1, quiet=True)
model_n2v_128 = node2vec_128.fit(window=10, min_count=1)

blue_emb_128 = np.array([
    team_embedding([row[f'blue_pick{i}'] for i in range(1,6)], model_n2v_128)
    for _, row in games.iterrows()
])
red_emb_128 = np.array([
    team_embedding([row[f'red_pick{i}'] for i in range(1,6)], model_n2v_128)
    for _, row in games.iterrows()
])

X_emb_128 = np.hstack([blue_emb_128, red_emb_128, blue_emb_128 - red_emb_128])
print('Shape:', X_emb_128.shape)

X_train_128, X_test_128, y_train_128, y_test_128 = train_test_split(
    X_emb_128, y, test_size=0.2, random_state=42
)

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_128, y_train_128)
preds = gb.predict(X_test_128)
print(f'GB + 128d embeddings + diff: Accuracy={accuracy_score(y_test_128, preds):.4f}, F1={f1_score(y_test_128, preds):.4f}')

Shape: (2087, 384)
GB + 128d embeddings + diff: Accuracy=0.5359, F1=0.6008


In [26]:
from sklearn.model_selection import cross_val_score

print('5-Fold Cross-Validation Results:')
print('='*55)

# baselines with one-hot
baseline_models = {
    'Logistic Regression (one-hot)': LogisticRegression(max_iter=1000),
    'Naive Bayes (one-hot)': BernoulliNB(),
    'Random Forest (one-hot)': RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in baseline_models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f'{name}: {scores.mean():.4f} +/- {scores.std():.4f}')

print('---')

# proposed model
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(gb, X_emb, y, cv=5, scoring='accuracy')
print(f'GB + Node2Vec Embeddings: {scores.mean():.4f} +/- {scores.std():.4f}')

5-Fold Cross-Validation Results:
Logistic Regression (one-hot): 0.4974 +/- 0.0293
Naive Bayes (one-hot): 0.4935 +/- 0.0045
Random Forest (one-hot): 0.4873 +/- 0.0117
---
GB + Node2Vec Embeddings: 0.5141 +/- 0.0172


In [27]:
from sklearn.model_selection import cross_val_score

print('Ablation Study - 5-Fold CV Accuracy')
print('='*55)

# 1. random embeddings (no graph structure, just random vectors)
np.random.seed(42)
random_emb = {champ: np.random.randn(64) for champ in mlb.classes_}

blue_rand = np.array([
    np.sum([random_emb[row[f'blue_pick{i}']] for i in range(1,6)], axis=0)
    for _, row in games.iterrows()
])
red_rand = np.array([
    np.sum([random_emb[row[f'red_pick{i}']] for i in range(1,6)], axis=0)
    for _, row in games.iterrows()
])
X_rand = np.hstack([blue_rand, red_rand])

scores = cross_val_score(GradientBoostingClassifier(n_estimators=100, random_state=42), X_rand, y, cv=5)
print(f'GB + Random Embeddings (no graph):  {scores.mean():.4f} +/- {scores.std():.4f}')

# 2. node2vec embeddings no graph weights (unweighted)
G_unweighted = nx.Graph()
for u, v in G.edges():
    G_unweighted.add_edge(u, v, weight=1)

n2v_unweighted = Node2Vec(G_unweighted, dimensions=64, walk_length=20, num_walks=200, workers=1, quiet=True)
model_unweighted = n2v_unweighted.fit(window=10, min_count=1)

blue_uw = np.array([
    team_embedding([row[f'blue_pick{i}'] for i in range(1,6)], model_unweighted)
    for _, row in games.iterrows()
])
red_uw = np.array([
    team_embedding([row[f'red_pick{i}'] for i in range(1,6)], model_unweighted)
    for _, row in games.iterrows()
])
X_uw = np.hstack([blue_uw, red_uw])

scores = cross_val_score(GradientBoostingClassifier(n_estimators=100, random_state=42), X_uw, y, cv=5)
print(f'GB + Node2Vec Unweighted graph:     {scores.mean():.4f} +/- {scores.std():.4f}')

# 3. full model (weighted graph)
scores = cross_val_score(GradientBoostingClassifier(n_estimators=100, random_state=42), X_emb, y, cv=5)
print(f'GB + Node2Vec Weighted graph (ours):{scores.mean():.4f} +/- {scores.std():.4f}')

# 4. swap GB for LR on top of embeddings
scores = cross_val_score(LogisticRegression(max_iter=1000), X_emb, y, cv=5)
print(f'LR + Node2Vec Weighted graph:       {scores.mean():.4f} +/- {scores.std():.4f}')

Ablation Study - 5-Fold CV Accuracy
GB + Random Embeddings (no graph):  0.5318 +/- 0.0210
GB + Node2Vec Unweighted graph:     0.5185 +/- 0.0182
GB + Node2Vec Weighted graph (ours):0.5141 +/- 0.0172
LR + Node2Vec Weighted graph:       0.5021 +/- 0.0185


In [28]:
from sklearn.model_selection import cross_val_score

print('Parameter Sensitivity - GB n_estimators')
print('='*45)
for n in [50, 100, 200, 300]:
    scores = cross_val_score(
        GradientBoostingClassifier(n_estimators=n, random_state=42),
        X_emb, y, cv=5
    )
    print(f'n_estimators={n}: {scores.mean():.4f} +/- {scores.std():.4f}')

print('\nParameter Sensitivity - GB learning rate')
print('='*45)
for lr in [0.01, 0.05, 0.1, 0.2]:
    scores = cross_val_score(
        GradientBoostingClassifier(learning_rate=lr, n_estimators=100, random_state=42),
        X_emb, y, cv=5
    )
    print(f'learning_rate={lr}: {scores.mean():.4f} +/- {scores.std():.4f}')

print('\nParameter Sensitivity - Node2Vec dimensions')
print('='*45)
for dim in [32, 64, 128]:
    n2v_temp = Node2Vec(G, dimensions=dim, walk_length=20, num_walks=200, workers=1, quiet=True)
    m_temp = n2v_temp.fit(window=10, min_count=1)
    blue_temp = np.array([
        team_embedding([row[f'blue_pick{i}'] for i in range(1,6)], m_temp)
        for _, row in games.iterrows()
    ])
    red_temp = np.array([
        team_embedding([row[f'red_pick{i}'] for i in range(1,6)], m_temp)
        for _, row in games.iterrows()
    ])
    X_temp = np.hstack([blue_temp, red_temp])
    scores = cross_val_score(
        GradientBoostingClassifier(n_estimators=100, random_state=42),
        X_temp, y, cv=5
    )
    print(f'dimensions={dim}: {scores.mean():.4f} +/- {scores.std():.4f}')

Parameter Sensitivity - GB n_estimators
n_estimators=50: 0.5194 +/- 0.0154
n_estimators=100: 0.5141 +/- 0.0172
n_estimators=200: 0.5093 +/- 0.0033
n_estimators=300: 0.4998 +/- 0.0126

Parameter Sensitivity - GB learning rate
learning_rate=0.01: 0.5223 +/- 0.0113
learning_rate=0.05: 0.5089 +/- 0.0224
learning_rate=0.1: 0.5141 +/- 0.0172
learning_rate=0.2: 0.5017 +/- 0.0186

Parameter Sensitivity - Node2Vec dimensions
dimensions=32: 0.5175 +/- 0.0209
dimensions=64: 0.5199 +/- 0.0234
dimensions=128: 0.5079 +/- 0.0265


In [29]:
best_model = GradientBoostingClassifier(
    n_estimators=50, 
    learning_rate=0.01, 
    random_state=42
)
scores = cross_val_score(best_model, X_emb, y, cv=5)
print(f'Final tuned model: {scores.mean():.4f} +/- {scores.std():.4f}')

# fit on full training data for the demo
best_model.fit(X_train_emb, y_train_emb)
preds = best_model.predict(X_test_emb)
print(f'Test accuracy: {accuracy_score(y_test_emb, preds):.4f}')
print(f'Test F1: {f1_score(y_test_emb, preds):.4f}')

Final tuned model: 0.5170 +/- 0.0145
Test accuracy: 0.4928
Test F1: 0.6431
